# Dependencies

In [ ]:
import os
import polars as pl
import glob
import json
import pandas as pd
import numpy as np
import math
import joblib

## Pre-Processing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import PCA

# Helper

In [ ]:
def get_all_file_names(source_folder_name:str,file_format:str = ""):
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format)>0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)


In [ ]:
def filter_file_names(list_file_names:list, filter_word:str):
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

In [ ]:
def to_pandas_dataframe(data, template_column: list) -> pd.DataFrame:
    df = pd.DataFrame(data)
    df = df.reindex(columns=template_column)
    return df

# Pre-Processing

## Change CSV to Parquet

In [ ]:
CHUNK_SIZE = 100000
LABEL_COLUMN = "Label"
CSV_FOLDER_NAME = "cse-cic-ids2018"
RAW_DATA_FOLDER_NAME = "raw"
COLUMNS_TO_DROP = {
    "flow id", "src ip", "source ip", "src port", "source port",
    "dst ip", "destination ip", "timestamp",
}

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [ ]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [ ]:
def dataframe_to_parquet(chunk: pd.DataFrame, target_folder_name: str, file_name: str, template_columns: list[str] | None = None):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

In [ ]:
def create_template_columns(files: list, source_folder_name: str):
    template_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        for c in cols:
            if c not in seen and c not in COLUMNS_TO_DROP:
                seen.add(c)
                template_columns.append(c)
    return template_columns

In [ ]:
def get_template_columns(source_folder_name, template_column_path: str = "dataframe-index.pkl"):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)
    joblib.dump(template_columns, template_column_path)

In [ ]:
def convert_all_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    for i, file_name in enumerate(csv_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        base_name = os.path.splitext(file_name)[0]

        for chunk_number, chunk in enumerate(pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1):
            dataframe_to_parquet(chunk, target_folder_name, f"{base_name}_{chunk_number:05d}.parquet", template_columns)

    print(f"\nFinished processing {total} files.")

In [ ]:
convert_all_file_to_parquet(CSV_FOLDER_NAME,RAW_DATA_FOLDER_NAME,CHUNK_SIZE)

## Exploratory Data Analysis

## Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [ ]:
NO_INF_DATA_FOLDER_NAME = "data-no-inf"

In [ ]:
def calculate_inf_values(
    source_folder_name: str
):
    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [ ]:
calculate_inf_values(RAW_DATA_FOLDER_NAME)

In [ ]:
def change_inf_to_nan(
    source_folder_name: str,
    target_folder_name: str
):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name,"parquet")

    total = len(parquet_files)

    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        target_file = os.path.join(target_folder_name, file_name)

        df = pd.read_parquet(source_file)

        df.replace([np.inf, -np.inf],np.nan,inplace=True)
        df.to_parquet(target_file,index=False)

    print(f"\nCompleted. Processed {total} files.")

In [ ]:
change_inf_to_nan(RAW_DATA_FOLDER_NAME, NO_INF_DATA_FOLDER_NAME)

In [ ]:
calculate_inf_values(NO_INF_DATA_FOLDER_NAME)

The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

```text
Completed. Processed 168 files.
Found 131799 inf values
```

**After cleaning:**

```text
Completed. Processed 168 files.
Found 0.0 inf values
```

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [ ]:
SPLIT_DATA_FOLDER_NAME = "data-split"
LABEL_SPLIT_FOLDER_NAME = "label-split"
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [ ]:
def create_train_dev_test_folder(source_folder_name: str, features_folder_name: str, labels_folder_name: str, target_day):

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    parquet_files = filter_file_names(parquet_files, target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    split_names = ("train", "dev", "test")
    feature_folders = {}
    label_folders = {}
    for split_name in split_names:
        feature_folder = os.path.join(features_folder_name, split_name)
        label_folder = os.path.join(labels_folder_name, split_name)
        os.makedirs(feature_folder, exist_ok=True)
        os.makedirs(label_folder, exist_ok=True)
        feature_folders[split_name] = feature_folder
        label_folders[split_name] = label_folder

    return parquet_files, feature_folders, label_folders

In [ ]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)

        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)

        train_df, temp_df = train_test_split(
            df,
            test_size=dev_size + test_size,
            random_state=random_state,
            stratify=df[label_column],
        )
        dev_df, test_df = train_test_split(
            temp_df,
            test_size=test_size / (dev_size + test_size),
            random_state=random_state,
            stratify=temp_df[label_column],
        )

        for split_name, split_df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
            labels = split_df[[label_column]]
            features = split_df.drop(columns=[label_column])

            features.to_parquet(os.path.join(feature_folders[split_name], file_name), index=False)
            labels.to_parquet(os.path.join(label_folders[split_name], file_name), index=False)

    print(f"\nCompleted splitting {total} files.")

In [ ]:
def split_a_single_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, feature_folders, label_folders = create_train_dev_test_folder(
        source_folder_name, features_folder_name, labels_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        feature_folders,
        label_folders,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, bruteforce)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, dos_golden)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, dos_hulk)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, ddos_http)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, ddos_udp)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, web_first)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, web_second)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, infiltration_first)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, infiltration_second)

In [ ]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME, SPLIT_DATA_FOLDER_NAME, LABEL_SPLIT_FOLDER_NAME, botnet)

Files are successfully split.

## Imputation

### Create Imputation

imputation for NaN

In [ ]:
def get_split_parquet_files(source_folder_name: str, split_name: str) -> list[str]:
    files = sorted(glob.glob(os.path.join(source_folder_name, split_name, "*.parquet")))
    if not files:
        raise FileNotFoundError(f"No Parquet files found in {os.path.join(source_folder_name, split_name)}")
    print(f"Found {len(files)} {split_name} files")
    return files

In [ ]:
def load_combined_lazyframe(train_files: list[str]) -> pl.LazyFrame:
    lazy_frames = []
    total = len(train_files)
    for i,file in enumerate(train_files,start=1):
        print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
        pl.scan_parquet(file)
    return pl.concat(lazy_frames, how="diagonal_relaxed")

In [ ]:
def get_numeric_columns(combined: pl.LazyFrame) -> list[str]:
    schema = combined.collect_schema()
    numeric_cols = [
        column for column, dtype in zip(schema.names(), schema.dtypes())
        if dtype.is_numeric()
    ]
    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [ ]:
def compute_medians(combined: pl.LazyFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    total = len(numeric_cols)
    for i,column in enumerate(numeric_cols,start=1):
        print(f"Processing [{i}/{total}] {column}...", end="\r", flush=True)
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs).collect()
    return medians

In [ ]:
def build_median_dict(medians: pl.DataFrame, numeric_cols: list[str]) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}")

    return median_dict

In [ ]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [ ]:
def get_median_imputation(source_folder_name: str, output_file: str) -> dict[str, float]:
    train_files = get_split_parquet_files(source_folder_name, "train")
    combined = load_combined_lazyframe(train_files)
    numeric_cols = get_numeric_columns(combined)
    medians = compute_medians(combined, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [ ]:
median = get_median_imputation(SPLIT_DATA_FOLDER_NAME, "median-imputer.json")
print(json.dumps(median, indent=4))

### Impute Training Data

In [ ]:
IMPUTED_DATA_FOLDER_NAME = "data-imputed"

In [ ]:
def impute_dataframe(df:pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [ ]:
def impute_all_split_files(source_folder_name: str, output_folder_name: str, medians_path: str = "median-imputer.json"):
    with open(medians_path, "r") as f:
        medians = json.load(f)

    for split_name in ("train", "dev", "test"):
        files = get_split_parquet_files(source_folder_name, split_name)
        total = len(files)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)

        for i, file in enumerate(files, start=1):
            print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
            df = pd.read_parquet(file)
            df = impute_dataframe(df, medians)
            output_file = os.path.join(output_split_folder, os.path.basename(file))
            df.to_parquet(
                output_file,
                engine="pyarrow",
                compression="snappy",
                index=False
            )

        print(f"\nFinished imputing {total} {split_name} files.")

In [ ]:
impute_all_split_files(SPLIT_DATA_FOLDER_NAME, IMPUTED_DATA_FOLDER_NAME)

## Normalized or Feature Scaling

### Create Scaler

In [ ]:
STANDARD_SCALER_PATH = "standard-scaler.pkl"

In [ ]:
def fit_scaler_on_files(scaler: StandardScaler, file_paths: list[str]) -> StandardScaler:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        scaler.partial_fit(df)
    return scaler

In [ ]:
def create_standard_scaler(source_folder_name: str, split_name: str = "train") -> StandardScaler:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    scaler = StandardScaler()
    scaler = fit_scaler_on_files(scaler, file_paths)

    if not hasattr(scaler, "mean_"):
        raise ValueError("The scaler could not be fitted because all files were empty.")

    print(" " * 100, end="\r")
    print(f"Successfully fitted scaler using {len(file_paths)} Parquet file(s) from '{split_name}'.")

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler,output_path:str):
    joblib.dump(scaler,output_path)

In [ ]:
def get_standard_scaler(source_folder_name:str,output_path:str):
    scaler = create_standard_scaler(source_folder_name)
    dump_standard_scaler(scaler,output_path)

In [ ]:
get_standard_scaler(IMPUTED_DATA_FOLDER_NAME, STANDARD_SCALER_PATH)

### Scale Training Data

In [ ]:
SCALED_DATA_FOLDER_NAME = "data-scaled"

In [ ]:
def load_standard_scaler(path:str)->StandardScaler:
    return joblib.load(path)

In [ ]:
def scale_split_files(source_folder_name: str, scaler_path: str, output_folder_name: str):
    scaler = load_standard_scaler(scaler_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        for i, file_path in enumerate(file_paths, start=1):
            print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
            df = pd.read_parquet(file_path)
            columns = df.columns.tolist()

            data = scaler.transform(df)
            scaled_df = pd.DataFrame(data, columns=columns)

            scaled_df.to_parquet(
                os.path.join(output_split_folder, os.path.basename(file_path)),
                engine="pyarrow",
                compression="snappy",
                index=False
            )

        print(f"\nFinished scaling {total} {split_name} files.")

In [ ]:
scale_split_files(IMPUTED_DATA_FOLDER_NAME, STANDARD_SCALER_PATH, SCALED_DATA_FOLDER_NAME)

# Principal Component Analysis (PCA)

### Incremental PCA

In [ ]:
IPCA_PATH = "ipca-transformer.pkl"

In [ ]:
def fit_ipca_on_files(ipca: IncrementalPCA, file_paths: list[str]) -> IncrementalPCA:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        ipca.partial_fit(df)

    return ipca

In [ ]:
def create_ipca(source_folder_name: str, split_name: str = "train") -> IncrementalPCA:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    ipca = IncrementalPCA()
    ipca = fit_ipca_on_files(ipca, file_paths)
    print(" " * 100, end="\r")
    print(f"Successfully fitted ipca using {len(file_paths)} Parquet file(s) from '{split_name}'.")

    return ipca

In [ ]:
def dump_ipca(ipca: IncrementalPCA,output_path:str):
    joblib.dump(ipca,output_path)

In [ ]:
def get_ipca(source_folder_name:str,output_path:str):
    ipca = create_ipca(source_folder_name)
    dump_ipca(ipca,output_path)

In [ ]:
get_ipca(SCALED_DATA_FOLDER_NAME,IPCA_PATH)

### PCA
use polars for efficiency